# NAC coloring search

In this notebook, we provide utilities to run benchmarks, analyze results, and experiment with our code.

First we provide functionality for loading graph classes,
then a simple function for measuring performance of listing all NAC-colorings on a graph class.
Then we define how strategies are passed to our algorithm,
and then a framework for defining and running benchmarks.
Lastly, we provide tools for results analysis.

Many utility functions (like LaTeX export) were moved from this notebook
into a separate file to improve clarity, see `benchmarks/notebook_utils.py`.

Make sure the `nac` directory is in your working directory, and that you installed `requirements.txt` into your virtual environment.
Also, make sure to extract files in `benchmarks/precomputed` if you want to analyze our prerun result yourself.

Please also refer to included `README.md` file.

In [ ]:
from typing import *
from collections import deque

import numpy as np
import pandas as pd
import networkx as nx
import os
import time
import itertools

from tqdm import tqdm

import nac as nac
from nac import NACValidClassType
import benchmarks
from benchmarks import datasets
from benchmarks.notebook_utils import *

seed=42

### Benchmarks directory

You can either choose to use our precomputed results or run the benchmarks yourself.
The algorithms take usually tens or hunderes of miliseconds to run,
but there is plenty of graphs and strategies combinations, so times add up.

In [ ]:
OUTPUT_DIR_PRECOMPUTED = os.path.join("benchmarks", "precomputed")
OUTPUT_DIR_LOCAL = os.path.join("benchmarks", "local")

benchmarks.notebook_utils.OUTPUT_DIR = OUTPUT_DIR_PRECOMPUTED
os.makedirs(benchmarks.notebook_utils.OUTPUT_DIR, exist_ok=True)

# Loading graph classes

In this section we load graphs that can be later used for running benchmarks.
The graphs are not in any specified order, and the datasets differ in size and graph sizes.
Graphs are stored in the `graph6` format in the `graphs_store` directory.

In [ ]:
class Graphs:
    """
    Randomly generated minimally rigid (Laman) graphs of various sizes.
    """
    minimally_rigid_random = LazyList(lambda: datasets.load_minimally_rigid_random_graphs())
    """
    Randomly generated globally rigid graphs using a threshold function.
    """
    globally_rigid_threshold = LazyList(lambda: datasets.load_globally_rigid_threshold_graphs())
    """
    Randomly generated NAC-critical graphs with at least n/4 △-connected components
    """
    nac_critical = LazyList(lambda: datasets.load_nac_critical_graphs())

    """
    Loads all the minimally rigid (Laman) graphs of the given size, pregenerated files allow the range of [5, 11]
    """
    def load_all_minimally_rigid(vertex_num: int) -> List[nx.Graph]:
        return list(datasets.load_minimally_rigid_all(vertices_num=vertex_num))

# Running on all minimally rigid graphs

This is a function that can be used for benchmarking of finding all the NAC-colorings of some graph class.
This function can provide only total times, not runtime per graph. That is the job of the following benchmarks.

In [ ]:
def benchmarks_all_NAC_coloring_on_class(
    graphs: List[nx.Graph],
    rounds: int,
    algorithm: str,
    use_nac_mono_classes: bool,
    use_has_coloring_check: bool = False,
) -> float:
    start = time.time()
    nac_mono_class_type=nac.NACValidClassType.EXTENDED if use_nac_mono_classes else nac.NACValidClassType.TRIANGLES

    for _ in range(rounds):
        for graph in graphs:
            # the fastest way to collect an iterable in Python
            deque(nac.NAC_colorings(
                graph,
                relabel_strategy="none",
                algorithm=algorithm,
                nac_valid_class_type=nac_mono_class_type,
                use_has_coloring_check=use_has_coloring_check,
            ), 0)

    return (time.time() - start) / rounds

def benchmarks_all_NAC_coloring_minimally_rigid(
    vertex_num: int,
    rounds: int,
):
    graphs = list(Graphs.load_all_minimally_rigid(vertex_num))
    for algorithm, use_triangle_extended_classes in [
        ('naive', False),
        ('naive', True),
        ('cycles', True),
        ('subgraphs-linear-neighbors_degree-4', True),
    ]:
        print(f"Vertices: {vertex_num}")
        print(f"Algorithm: {algorithm}")
        print(f"△-extended classes: {"enabled" if use_triangle_extended_classes else "disabled"}")
        runtime = benchmarks_all_NAC_coloring_on_class(
            graphs=graphs,
            rounds=rounds,
            algorithm=algorithm,
            use_nac_mono_classes=use_triangle_extended_classes,
        )
        print(f"Runtime: {runtime:.3f} s")
        print()

if False: # Change to enable
    for n in range(5, 11+1):
        benchmarks_all_NAC_coloring_minimally_rigid(
            vertex_num=n,
            rounds=3 if n <= 10 else 1,
        )

# Storing and loading benchmark results

We store records of graph's results in CSV files.
Each row represents performance of a graph with a given strategy.
The difference between the variants `first` and `all` is
whether a single or all NAC-colorings are searched.

The export CSV columns are:
- `timestamp` - date time of the test in UTC
- `graph` - base64 encoded bytes of graph6 encoded graph
- `dataset` - class of the graph like `minimally_ridig_random`, `no_3_nor_4_cycles`, `globally_rigid`, ...
- `vertex_num` - the number of vertices of the graph
- `edge_num` - the number of edges of the graph
- `triangle_components_num` - the number of $\triangle$ components of the graph
- `extended_classes_num` - the number of $\triangle$-extended classes of the graph
- `relabel` - relabel strategy (relabels vertices before the main algorithm is run, here we have only `none` or `random`)
- `split` - splitting strategy
- `merge` - merging strategy
- `subgraph_size` - the target initial size of subgraphs in NAC-mono components
- `used_extended_classes` - if $\triangle$-extended classes were used to run the test, `False` means $\triangle$-connected components were used
- `nac_any_finished` - if any of the tests finished in time
- `nac_any_timeout_milliseconds` - timeout used for a single round during the benchmark run
- `nac_{first|all}_coloring_num` - the number of NAC-colorings of the graph, for the first variant limited to 1
- `nac_{first|all}_mean_time` - the time required to find first/all NAC-colorings in milliseconds
- `nac_{first|all}_rounds` - the number of rounds used to run the benchmarks
- `nac_{first|all}_check_cycle_mask` - the number of cycle mask checks performed
- `nac_{first|all}_check_is_NAC` - the number of `IsNACColoring` checks performed
- `nac_{first|all}_merge` - the number of merges performed
- `nac_{first|all}_merge_no_common_vertex` - the number of merges with no common vertex (these are simple to compute, but produce large number of colorings slowing down the algorithm)

In [ ]:
display(COLUMNS)

# Strategies

The interface of the NAC-coloring search function looks like this:
```python
def NAC_colorings(
    graph: nx.Graph,
    algorithm: str = "subgraphs",
    relabel_strategy: str = "none",
    nac_valid_class_type: NACValidClassType = NACValidClassType.EXTENDED,
    use_decompositions: bool = True,
    use_has_coloring_check: bool = True,
    seed: int | None = None,
) -> Iterable[NACColoring]:
```

`NACValidClassType` types are `EXTENDED` that creates $\triangle$-extended classes as described in the paper,
`TRIANGLES` that finds only $\triangle$-connected components and
`EDGES` that uses no NAC-valid classes optimization.
The `use_decompositions` switch is responsible for enabling checks for articulation points and related decomposition into blocks.
The `use_has_coloring_check` runs some polynomial checks if a NAC-coloring can exist. If not, the whole search is skipped.
`seed` is used by strategies internally as only pseudo random number generators are used.

The most important field is the `algorithm` field.
Possible values are:
- `"naive"` - runs naive algorithm
- `"cycles"` - runs naive algorithm improved by cycles detection
- `"subgraphs"` - runs so far optimal algorithm for larger graphs based on subgraph decomposition
- `"subgraphs-{merge_strategy}-{split_strategy}-{size_of_subgraphs}"` - runs the specified strategy combination with subgraph decomposition

In our code strategies are represented as three-tuples.

In [ ]:
class Promising:
    SPLITTING = [
        "none",
        "cycles_match_chunks",
        "neighbors",
        "neighbors_degree",
    ]
    MERGE = [
        "linear",
        "shared_vertices",
    ]
    SIZES = [6]

    strategies = list(itertools.product(
        SPLITTING, MERGE, SIZES,
    ))
print(f"Strategies: {len(Promising.strategies)}")

In [ ]:
def construct_subgraph_algo_name(param: Tuple[str, str, int]) -> str:
    split, merge, subgraph = param
    algo_name = "subgraphs-{}-{}-{}".format( merge, split, subgraph)
    return algo_name

In case you want to play with the notebook,
 we predefined some strategies (algorithm names) as example.

In [ ]:
ALGO_NONE_LINEAR = construct_subgraph_algo_name(("none", "linear", 6))
ALGO_NEIGHBORS_LINEAR = construct_subgraph_algo_name(("neighbors", "linear", 6))
ALGO_NEIGHBORS_DEGREE_LINEAR = construct_subgraph_algo_name(("neighbors_degree", "linear", 6))
ALGO_NEIGHBORS_DEGREE_SHARED_VERTICES = construct_subgraph_algo_name(("neighbors_degree", "shared_vertices", 6))

display([ALGO_NONE_LINEAR, ALGO_NEIGHBORS_LINEAR, ALGO_NEIGHBORS_DEGREE_LINEAR, ALGO_NEIGHBORS_DEGREE_SHARED_VERTICES])
display(list(nac.NAC_colorings(
    graph=nx.path_graph(4),
    algorithm=ALGO_NEIGHBORS_DEGREE_LINEAR,
)))

### Running and recording benchmarks

This cell serves as the main interface for running benchmarks.
Note that benchmarks that were already run are skipped on purpose as long as precomputed data are available.
For parameters explanation, see the doc string bellow.

In [ ]:
def measure_for_graph_class(
    dataset_name: str,
    graphs: Iterable[nx.Graph],
    graph_timeout: int,
    all_max_classes_num: int = 28,
    rounds: int = 2,
    allow_naive: bool = True,
    use_triangle_extended_classes: bool = True,
    df_seen: pd.DataFrame | Callable[[], pd.DataFrame] |None = lambda: load_records(),
    save_every: int | None = 5*60,
    verbose: bool = False,
) -> pd.DataFrame:
    """
    Runs benchmarks for the given graph class.

    Parameters:
        dataset_name: Name of the dataset stored in the output csv
        graphs: Iterable of graphs to benchmark
        graph_timeout: Timeout for each graph in seconds
        all_max_classes_num: Maximum number of NAC-valid classes (based on use_triangle_extended_classes) to search for all NAC-colorings
        rounds: Number of rounds to run for each graph
        allow_naive: Whether to run the naive algorithm
        use_smart_split: Whether to use smart split
        use_triangle_extended_classes: Whether to use △-extended classes or △-connected components
        df_seen: Dataframe with already measured data, so already tried graphs and strategies can be skipped
        save_every: save progress every number of seconds
        verbose: print Timings for individual strategies
    """
    if callable(df_seen):
        df_seen = df_seen()

    dataset_name = dataset_name.replace(" ", "_").lower()
    if df_seen is None:
        df_seen = to_benchmark_results()
    df_seen = df_seen.query(f"dataset == '{dataset_name}'")
    df_seen.set_index("graph", inplace=True)

    results: List[MeasurementResult] = []
    all_results: List[MeasurementResult] = []

    last_save = time.time()

    for graph in tqdm(graphs):
        if save_every is not None:
            now = time.time()
            if now - last_save > save_every:
                all_results.extend(results)
                df = to_benchmark_results(results)
                update_stored_data([df], head_loaded=False)
                results = []
                last_save = now


        triangle_component_num = len(nac.find_nac_mono_classes(graph=graph, class_type=NACValidClassType.TRIANGLES)[1])
        extended_classes_num = len(nac.find_nac_mono_classes(graph=graph, class_type=NACValidClassType.EXTENDED)[1])
        if use_triangle_extended_classes:
            first_only = all_max_classes_num < extended_classes_num
        else:
            first_only = all_max_classes_num < triangle_component_num

        strategies = Promising.strategies
        if allow_naive:
            strategies = itertools.chain(strategies, (None,))

        graph_id = graph_to_id(graph)
        if graph_id in df_seen.index:
            df_graph = df_seen.loc[graph_id]
        else:
            df_graph = df_seen.iloc[:0]

        for strategy in strategies:
            # skip test that already run
            relabel = "none"
            if strategy is not None:
                algorithm = construct_subgraph_algo_name(strategy)
                prev_record = df_graph.query(
                    f"{Columns.SPLIT} == '{strategy[0]}'"
                    + f" and {Columns.MERGING} == '{strategy[1]}'"
                    + f" and {Columns.SUBGRAPH_SIZE} == {strategy[2]}"
                    + f" and {Columns.USED_EXTENDED_CLASSES} == {use_triangle_extended_classes}"
                )
            else:
                algorithm = "cycles"
                prev_record = df_graph.query(
                    f"{Columns.SPLIT} == 'naive-cycles'"
                    + f" and {Columns.MERGING} == 'naive-cycles'"
                    + f" and {Columns.SUBGRAPH_SIZE} == 0"
                    + f" and {Columns.USED_EXTENDED_CLASSES} == {use_triangle_extended_classes}"
                )
            if len(prev_record) > 0:
                if first_only or list(prev_record[Columns.ALL_MEAN_TIME])[-1] > 0:
                    continue

            try:
                search_res = nac_benchmark_core(
                    graph,
                    rounds=rounds,
                    first_only=first_only,
                    algorithm=algorithm,
                    relabel_strategy=relabel,
                    use_triangle_extended_classes=use_triangle_extended_classes,
                    time_limit=graph_timeout,
                )

                split, merge, subgraph_size = strategy if strategy else ("naive-cycles", "naive-cycles", 0)
                res = create_measurement_result(
                    graph=graph,
                    dataset_name=dataset_name,
                    triangle_components=triangle_component_num,
                    extended_classes=extended_classes_num,
                    nac_first=search_res.first,
                    nac_all=search_res.all,
                    relabel_strategy=relabel,
                    split_strategy=split,
                    merge_strategy=merge,
                    subgraph_size=subgraph_size,
                    use_smart_split=False,
                    used_triangle_extended_classes=use_triangle_extended_classes,
                    timeout_milliseconds=graph_timeout * 1000,
                )
                results.append(res)
                if (verbose):
                    print(f"Strategy {strategy} took {res.nac_first_mean_time} ms")
            except Exception as e:
                print("Exception:", e)

    all_results.extend(results)
    if len(all_results) == 0:
        print("All runs skipped")

    df = to_benchmark_results(results)
    update_stored_data([df], head_loaded=False)

    df = to_benchmark_results(all_results)
    df = df.sort_values(by=[Columns.ALL_MEAN_TIME, Columns.FIRST_MEAN_TIME])
    return df

# Running benchmarks

You can run any of these cells by changing the condition and running the cell. As described above, benchmarks take long to run because each graph is run 2 times for each enabled strategy.
There is autosave enabled that stores progress every 5 minutes.
Do not forget to change `OUTPUT_DIR` at the beginning of the notebook otherwise the tests will be skipped as they are already precomputed for you.

We use *rigid* instead of *2-rigid* in this notebook.

In [ ]:
if False:
    measure_for_graph_class(
        "Minimally rigid random",
        Graphs.minimally_rigid_random,
        graph_timeout=3,
    )

### Globally rigid

Randomly generated globally rigid graphs.

In [ ]:
if False:
    measure_for_graph_class(
        "Globally rigid threshold",
        Graphs.globally_rigid_threshold,
        graph_timeout=5,
    )

### NAC critical

Random graphs generated using the threshold function for NAC-coloring existence.

In [ ]:
if False:
    measure_for_graph_class(
        "NAC critical",
        Graphs.nac_critical,
        graph_timeout=5,
        all_max_classes_num=0,
        use_triangle_extended_classes=False,
        allow_naive=False,
    )

# Analytics

In this section, we provide a framework for plotting results of the previous benchmarks.

All the charts plotted bellow in this section are created from runs with more than one △-extended class or △-connected component based on the setting.
Otherwise, the results can be obtained immediately as the answer is trivial.
Therefore, we filter them out.

The first group of graphs show the time required to find
a first/all NAC-colorings based on the number of vertices or the number of NAC-mono classes.
In one row you can see mean, median and 3rd quartile plots with lines for each strategy.

In [ ]:
df_analytics_loaded = load_records()
df_analytics_loaded.set_index(Columns.GRAPH, inplace=True)
df_analytics_loaded = df_analytics_loaded.query(f"{Columns.DATASET} != 'test'")
display(df_analytics_loaded.columns)
display(list(df_analytics_loaded[Columns.DATASET].unique()))
display(list(df_analytics_loaded[Columns.RELABEL].unique()))
display(list(df_analytics_loaded[Columns.SPLIT].unique()))
display(list(df_analytics_loaded[Columns.MERGING].unique()))

# Transform
df_analytics_loaded = df_analytics_loaded.assign(split_merging=lambda x: (x[Columns.SPLIT] + " & " + x[Columns.MERGING]).str.replace("naive-cycles & naive-cycles", "naive cycles"))
df_analytics_loaded = df_analytics_loaded.assign(split_merging_smart=lambda x: x["split_merging"] + " & " + x[Columns.USE_SMART_SPLIT].astype(str))
df_analytics_loaded.sort_values(by=Columns.SPLIT, inplace=True, kind="stable") # to make graph colors more regular
df_analytics_loaded.sort_values(by=Columns.MERGING, inplace=True, kind="stable")

In [ ]:
# Preserver the original data
df_analytics = df_analytics_loaded

# Filter out trivial graphs
df_analytics = df_analytics.query(f"({Columns.EXTENDED_CLASSES_NUM} > 1 and {Columns.USED_EXTENDED_CLASSES} == True) or ({Columns.TRIANGLE_COMPONENTS_NUM} > 1 and {Columns.USED_EXTENDED_CLASSES} == False)")

# Graphs with no NAC coloring and more △-connected components
df_analytics_no_nac = df_analytics.query(f"{Columns.FIRST_COLORING_NUM} == 0 and {Columns.TRIANGLE_COMPONENTS_NUM} > 1 and {Columns.USED_EXTENDED_CLASSES} == False")

# Statistics
def analyze_general(df: pd.DataFrame) -> None:
    print(f"Total runs: {len(df)}", )
    print(f"Total graphs: {len(df.index.unique())}")
    df_finished = df.query(f"{Columns.ANY_FINISHED} == True")
    df_failed = df.query(f"{Columns.ANY_FINISHED} == False")
    print(f"Runs that did/not finish: {len(df_finished)}/{len(df_failed)} ({np.round(len(df_failed)/len(df)*100, 1)}% did not finish)")
    print(f"Graphs where some runs did/not finish: {df_finished.index.nunique()}/{df_failed.index.nunique()}")

def analyze_colorings(df: pd.DataFrame) -> None:
    df_finished = df.query(f"{Columns.ANY_FINISHED} == True")
    print(f"Graphs with  a NAC-coloring:", df_finished.query(f"{Columns.FIRST_COLORING_NUM}  > 0").index.nunique())
    print(f"Graphs with no NAC-coloring:", df_finished.query(f"{Columns.FIRST_COLORING_NUM} == 0").index.nunique())
    print(f"Graphs with no NAC-coloring and more △-extended classes:", df_finished.query(f"{Columns.FIRST_COLORING_NUM} == 0 and {Columns.EXTENDED_CLASSES_NUM} > 1").index.nunique())

def analyze_finished(df: pd.DataFrame) -> None:
    graphs = df.index.unique()
    graphs_all_finished = filter_graphs_that_finished_for_all_strategies(df)
    graphs_nonnaive_finished = filter_graphs_that_finished_for_all_strategies(df.query(f"{Columns.SPLIT} != 'naive-cycles'"))
    print(f"{len(graphs_all_finished)}/{len(graphs)} graphs finished on all tested strategies.")
    print(f"{len(graphs_nonnaive_finished)}/{len(graphs)} graphs finished on all tested strategies excluding naive cycles.")

    df_all_finished = df.loc[graphs_all_finished]
    df_nonnaive_finished = df.loc[graphs_nonnaive_finished]
    print(f"Records corresponding to graph, that finished on all tested strategies: {len(df_all_finished)}")
    print(f"Records corresponding to graph, that finished on all tested strategies excluding naive cycles: {len(df_nonnaive_finished)}")

print("Just preffered strategies:")
analyze_general(df_analytics)
analyze_colorings(df_analytics)
print()

analyze_finished(df_analytics)

df_analytics = replace_failed_results(df_analytics)

### Ratio between $\triangle$-extended and $\triangle$-connected components.

In [ ]:
for dataset in ["nac_critical", "globally_rigid_threshold", "minimally_rigid_random"]:
    fig = plot_extended_vs_original_triangle_components(df_analytics, dataset=dataset)
    display(fig)

### Minimally rigid - Random

Randomly generated minimally rigid graphs of various sizes.

In [ ]:
if True:
    title = 'Minimally rigid'
    dataset_name = 'minimally_rigid_random'

    dataset = df_analytics.query(f"{Columns.DATASET} == '{dataset_name}'")
    figs = [fig for fig in plot_frame(title, dataset, ops_value_columns_sets=Columns.first)]
    [display(fig) for fig in figs]

    dataset = df_analytics.query(f"{Columns.DATASET} == '{dataset_name}'")
    figs = [fig for fig in plot_frame(title, dataset, ops_value_columns_sets=Columns.all)]
    [display(fig) for fig in figs]

### Globally rigid graphs

Randomly generated globally rigid graphs.

In [ ]:
if True:
    title = 'Globally rigid'
    dataset_name = 'globally_rigid_threshold'

    dataset = finished_graphs(df_analytics.query(
        f"{Columns.DATASET} == '{dataset_name}' and {Columns.EXTENDED_CLASSES_NUM} <= 28")
    )
    figs = [fig for fig in plot_frame(title, dataset,
        ops_value_columns_sets=Columns.first,
    )]
    [display(fig) for fig in figs]

    dataset = finished_graphs_no_naive(df_analytics.query(f"{Columns.DATASET} == '{dataset_name}'"))
    figs = [fig for fig in plot_frame(title, dataset,
        ops_value_columns_sets=Columns.all,
        ops_x_column=[Columns.EXTENDED_CLASSES_NUM],
    )]
    [display(fig) for fig in figs]

### NAC-critical

Random graphs generated using the threshold function for NAC-coloring existence.

In [ ]:
if True:
    dataset_name = 'nac_critical'

    title = 'NAC-critical - no NAC coloring'
    dataset = finished_graphs(df_analytics.query(f"{Columns.DATASET} == '{dataset_name}' and {Columns.FIRST_COLORING_NUM} == 0"))
    figs = [fig for fig in plot_frame(title, dataset,
        ops_value_columns_sets=Columns.first,
        ops_x_column=[Columns.TRIANGLE_COMPONENTS_NUM],
    )]
    [display(fig) for fig in figs]

    title = 'NAC-critical - some NAC coloring'
    dataset = finished_graphs(df_analytics.query(f"{Columns.DATASET} == '{dataset_name}' and {Columns.FIRST_COLORING_NUM} > 0"))
    figs = [fig for fig in plot_frame(title, dataset,
        ops_value_columns_sets=Columns.first,
        ops_x_column=[Columns.TRIANGLE_COMPONENTS_NUM],
    )]
    [display(fig) for fig in figs]

## The number of checks needed

This group of graphs compares the number of `IsNACColoring` checks
performed by our algorithm and by naive algorithm
using either no NAC-valid classes (just edges),
$\triangle$-components or $\triangle$-extended classes
as described in our article.

It is expected that the number of `IsNACColoring` checks
will be smaller than the `CycleMask` checks (checks for small cycles using bit-mask, Section 4.2)
as the `CycleMask` checks happen every time,
but `IsNACColoring` checks happen only if the previous checks fail.

Unless you change anything, the result is plotted
from the whole benchmarking dataset - all the graphs classes are used.
You can add `query("dataset == '...'")` to show the graph for a specific dataset.

In [ ]:
if True:
    figs = [fig for fig in plot_is_NAC_coloring_calls(df_analytics.query(
        f"({Columns.SPLIT} != 'naive-cycles') and ({Columns.USED_EXTENDED_CLASSES}==True) and {Columns.EXTENDED_CLASSES_NUM} <= 28")
    )]
    title = 'All datasets'
    dataset_name = 'check-comparision'
    [display(fig) for fig in figs]
    export_standard_figure_list(dataset_name, figs)